In [17]:
import random
import copy

## CARD Class And Deck Generator
class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = value # '0-9' or 'skip'
        
    def __repr__(self):
        return f"({self.color} {self.value})"
        
    def __eq__(self, other):
        if isinstance(other, Card):
            return self.color == other.color and self.value == other.value
        return False

def generateDeck():
    colors = ['red', 'blue', 'green', 'yellow']
    values = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'skip']
    deck = [Card(c, v) for c in colors for v in values]
    random.shuffle(deck)
    return deck  

## State Representation and Transitions
class UnoState:
    def __init__(self, hands, topCard, deck, current_player = 0):
        self.hands = hands # it is a list of 3 lists representing 3 players
        self.deck = deck
        self.current_player = current_player
        self.topCard = topCard
    
    def getvalidmoves(self, playeridx):
        hand = self.hands[playeridx] # first fetch the cards of required player
        moves = [card for card in hand if card.color == self.topCard.color or card.value == self.topCard.value]
        return moves

    def applyMove(self, move):
        newstate = copy.deepcopy(self)
        p_idx = newstate.current_player

        if move == 'draw':
            if newstate.deck:
                drawn_card = newstate.deck.pop(0)
                newstate.hands[p_idx].append(drawn_card)
            newstate.current_player = (p_idx + 1) % 3
        else:
            newstate.hands[p_idx].remove(move)
            newstate.topCard = move
            if move.value == 'skip':
                newstate.current_player = (p_idx + 2) % 3 #skip next player
            else:
                newstate.current_player = (p_idx + 1) % 3
        return newstate

    def is_terminal(self):
        return any(len(hand) == 0 for hand in self.hands) or len(self.deck) == 0

## Evaluation Functions
def evaluateState(state, player_idx, strategy):
    """
    Base Formula: Score = 50 - 5(C_AI) + 2(C_opp) + 3(S)
    """
    my_hand = state.hands[player_idx]
    opp1_hand = state.hands[(player_idx + 1) % 3]
    opp2_hand = state.hands[(player_idx + 2) % 3]
    c_ai = len(my_hand)
    c_opp = (len(opp1_hand) + len(opp2_hand)) / 2.0
    s_cards = sum(1 for card in my_hand if card.value == 'skip')

    if strategy == 'Defensive':
        # Defensive (Minimax P1): Prioritize hoarding skips to stall, heavily penalize opponents getting low cards
        score = 50 - 3 * c_ai + 5 * c_opp + 8 * s_cards 
    elif strategy == 'Offensive':
        # Offensive (Expectimax P2): Aggressive shedding, heavily penalize own hand size
        score = 50 - 8 * c_ai + 1 * c_opp + 2 * s_cards
    else:
        score = 50 - 5 * c_ai + 2 * c_opp + 3 * s_cards # Default baseline
    
    if c_ai == 0:
        score += 1000 #win condition

    return score

In [21]:
def minimax(state, depth, maximizing_player_idx):
    if depth == 0 or state.is_terminal():
        return evaluateState(state, maximizing_player_idx, 'Defensive'), None

    current_player = state.current_player
    valid_moves = state.getvalidmoves(current_player)
    
    if not valid_moves:
        valid_moves = ['draw']

    best_move = None

    if current_player == maximizing_player_idx:
        max_eval = float('-inf')
        for move in valid_moves:
            child_state = state.applyMove(move)
            eval_val, _ = minimax(child_state, depth - 1, maximizing_player_idx)
            if eval_val > max_eval:
                max_eval = eval_val
                best_move = move
        return max_eval, best_move
    else:
        # Assuming opponents play optimally against us (minimizing our score)
        min_eval = float('inf')
        for move in valid_moves:
            child_state = state.applyMove(move)
            eval_val, _ = minimax(child_state, depth - 1, maximizing_player_idx)
            if eval_val < min_eval:
                min_eval = eval_val
                best_move = move
        return min_eval, best_move

def expectimax(state, depth, maximizing_player_idx):
    if depth == 0 or state.is_terminal():
        return evaluateState(state, maximizing_player_idx, 'Offensive'), None

    current_player = state.current_player
    valid_moves = state.getvalidmoves(current_player)
    
    # CHANCE NODE: When a player must draw
    if not valid_moves:
        expected_value = 0
        if not state.deck:
            return evaluateState(state, maximizing_player_idx, 'Offensive'), 'draw'
        
        # To prevent exponential branching timeout, we sample a subset of the deck to calculate expectation
        sample_size = min(5, len(state.deck))
        prob = 1.0 / sample_size
        
        for i in range(sample_size):
            child_state = copy.deepcopy(state)
            drawn_card = child_state.deck.pop(i) # Simulating drawing different cards
            child_state.hands[current_player].append(drawn_card)
            child_state.current_player = (current_player + 1) % 3
            
            eval_val, _ = expectimax(child_state, depth - 1, maximizing_player_idx)
            expected_value += prob * eval_val 
            
        return expected_value, 'draw'

    best_move = None

    if current_player == maximizing_player_idx:
        # MAX node -> AI turn 
        max_eval = float('-inf')
        for move in valid_moves:
            child_state = state.applyMove(move)
            eval_val, _ = expectimax(child_state, depth - 1, maximizing_player_idx)
            if eval_val > max_eval:
                max_eval = eval_val
                best_move = move
        return max_eval, best_move
    else:
        # Opponent node -> opponent turn
        # Opponent chooses a random legal move 
        expected_value = 0
        prob = 1.0 / len(valid_moves)
        for move in valid_moves:
            child_state = state.applyMove(move)
            eval_val, _ = expectimax(child_state, depth - 1, maximizing_player_idx)
            expected_value += prob * eval_val
        
        return expected_value, valid_moves[0] # Return expected average, specific move matters less here

In [22]:
def play_game():
    deck = generateDeck()
    
    # Each player gets 5 cards
    hands = [[deck.pop(0) for _ in range(5)] for _ in range(3)]
    
    topCard = deck.pop(0)
    while topCard.value == 'skip': # Avoid starting with a skip to prevent edge-case turn skipping at the root
        deck.append(topCard)
        random.shuffle(deck)
        topCard = deck.pop(0)

    state = UnoState(hands, topCard, deck)
    
    print("--- STARTING GAME ---")
    
    while not state.is_terminal():
        p_idx = state.current_player
        print(f"\nTop card: {state.topCard}") 
        print("AI hand:")
        for card in state.hands[p_idx]:
            print(f"  {card}")
            
        print(f"AI decision (Depth 3):") 
        
        if p_idx == 0:
            print(">> Player 1 (Minimax - Defensive) is thinking...")
            score, move = minimax(state, 3, 0)
        elif p_idx == 1:
            print(">> Player 2 (Expectimax - Offensive) is thinking...")
            score, move = expectimax(state, 3, 1)
        elif p_idx == 2:
            print(">> Player 3 (Simulation Mode Minimax) is thinking...")
            score, move = minimax(state, 3, 2)

        if move == 'draw':
            print(f"Draw Card")
            print(f"Expected score: {score:.2f}") 
        else:
            print(f"Play: {move}") 
            print(f"Expected score: {score:.2f}") 
            
        state = state.applyMove(move)

    print("\n--- GAME OVER ---")
    for i in range(3):
        if len(state.hands[i]) == 0:
            print(f"Player {i+1} won!")

if __name__ == "__main__":
    play_game()

--- STARTING GAME ---

Top card: (red 7)
AI hand:
  (yellow skip)
  (blue 7)
  (blue skip)
  (red skip)
  (green skip)
AI decision (Depth 3):
>> Player 1 (Minimax - Defensive) is thinking...
Play: (blue 7)
Expected score: 90.00

Top card: (blue 7)
AI hand:
  (red 3)
  (yellow 0)
  (green 7)
  (yellow 7)
  (blue 3)
AI decision (Depth 3):
>> Player 2 (Expectimax - Offensive) is thinking...
Play: (green 7)
Expected score: 21.50

Top card: (green 7)
AI hand:
  (blue 5)
  (green 5)
  (green 2)
  (yellow 4)
  (yellow 9)
AI decision (Depth 3):
>> Player 3 (Simulation Mode Minimax) is thinking...
Play: (green 5)
Expected score: 58.50

Top card: (green 5)
AI hand:
  (yellow skip)
  (blue skip)
  (red skip)
  (green skip)
AI decision (Depth 3):
>> Player 1 (Minimax - Defensive) is thinking...
Play: (green skip)
Expected score: 79.50

Top card: (green skip)
AI hand:
  (blue 5)
  (green 2)
  (yellow 4)
  (yellow 9)
AI decision (Depth 3):
>> Player 3 (Simulation Mode Minimax) is thinking...
Play: (